In [1]:
from pathlib import Path
import sys
import warnings
import os
import gc
from copy import deepcopy

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.ops import MLP
import torch.optim as optim
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, StratifiedKFold
from torch.utils.data import Dataset, TensorDataset, DataLoader, Subset
import optuna
import joblib
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import random
import math
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns
import anndata
from anndata import AnnData
import pickle

sys.path.insert(0, "../../")

import scgpt as scg
from scgpt.tokenizer import GeneVocab
from scgpt import logger

warnings.filterwarnings("ignore", category=ResourceWarning)

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    
seed = 13
seed_everything(seed)

/home/harshil.sharma/miniconda3/envs/gene2ephys/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/allen/programs/mindscope/workgroups/auto-model/harshil.sharma/TransformerEphysPrediction/Human/w_types_comparison/../../scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/allen/programs/mindscope/workgroups/auto-model/harshil.sharma/TransformerEphysPrediction/Human/w_types_comparison/../../scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


In [2]:
cell_embeddings_adata = anndata.read_h5ad("../Data Preprocessing/cell_embeddings_adata.h5ad")
combined_ephys_df = pd.read_pickle("../Data Preprocessing/combined_ephys_df.pkl")

scgpt_embeddings_tensor = torch.tensor(cell_embeddings_adata.obsm["X_scGPT"], dtype=torch.float32)
hvg_expression_tensor = torch.tensor(cell_embeddings_adata[:, cell_embeddings_adata.var["highly_variable"]].X, dtype=torch.float32)
ionchannel_expression_tensor = torch.tensor(cell_embeddings_adata[:, cell_embeddings_adata.var["Ion channel coding"]].X, dtype=torch.float32)
cell_type_tensor = torch.tensor(cell_embeddings_adata.obsm["OneHotCellType"], dtype=torch.float32)

ephys_tensor = torch.tensor(combined_ephys_df.drop(columns=["SpecimenID"]).values, dtype=torch.float32)

embeddings_withtype_tensor = torch.cat((scgpt_embeddings_tensor, cell_type_tensor), dim=1)
hvg_expression__withtype_tensor = torch.cat((hvg_expression_tensor, cell_type_tensor), dim=1)
ionchannel_expression_withtype_tensor = torch.cat((ionchannel_expression_tensor, cell_type_tensor), dim=1)

scgpt_dataset = TensorDataset(embeddings_withtype_tensor, ephys_tensor)
hvg_dataset = TensorDataset(hvg_expression__withtype_tensor, ephys_tensor)
ionchannel_dataset = TensorDataset(ionchannel_expression_withtype_tensor, ephys_tensor)
datasets = {"scgpt": scgpt_dataset, "hvg": hvg_dataset, "ion": ionchannel_dataset}

In [3]:
def train_test_MLP(input_layer_size, train_dataloader, test_dataloader, num_epochs, lr, dropout, weight_decay, trial=None, step_offset=0):
    model = MLP(input_layer_size, [512, 512, 8], activation_layer=nn.Tanh, dropout=dropout)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    train_losses = []
    test_losses = []
    r2_scores = []

    for epoch in range(num_epochs):
        # training
        model.train()
        train_loss = torch.zeros((), device=device)
        for data_batch, targets_batch in train_dataloader:
            data_batch, targets_batch = data_batch.to(device), targets_batch.to(device)
            optimizer.zero_grad()
            outputs = model(data_batch)
            loss = criterion(outputs, targets_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.detach()*data_batch.size(0)
        train_loss = (train_loss / len(train_dataloader.sampler)).item()
        train_losses.append(train_loss)

        # testing
        model.eval()
        test_loss = torch.zeros((), device=device)
        all_outputs = []
        all_targets = []
        with torch.no_grad():
            for data_batch, targets_batch in test_dataloader:
                data_batch, targets_batch = data_batch.to(device), targets_batch.to(device)
                outputs = model(data_batch)
                loss = criterion(outputs, targets_batch)
                test_loss += loss.detach()*data_batch.size(0)
                all_outputs.append(outputs)
                all_targets.append(targets_batch)
            test_loss = (test_loss / len(test_dataloader.sampler)).item()
            test_losses.append(test_loss)

        if trial is None:  # these return values only matter when not tuning
            all_outputs = torch.cat(all_outputs, dim=0)
            all_targets = torch.cat(all_targets, dim=0)
            all_outputs, all_targets = all_outputs.cpu(), all_targets.cpu()
            r2_scores.append(r2_score(all_targets, all_outputs, multioutput="raw_values")) # r2_scores is a list of num_epochs numpy arrays each 8 long (for each ephys feature)

        # step_offset gives each inner CV fold its own, non-overlapping step range so MedianPruner compares trials at matching steps/epochs
        if trial is not None:
            trial.report(test_loss, step_offset + epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        
    return model, all_outputs, all_targets, train_losses, test_losses, r2_scores

# for nested cross-validation loop
def make_objective(trainval_idx, dataset, input_layer_size, num_epochs, batch_size, seed):
    def objective(trial):
        # hyperparameter space
        lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)
        weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
        inner_scores = []

        labels = cell_embeddings_adata.obs["CellType"].iloc[trainval_idx].values

        for inner_fold, (sub_train_idx, val_idx) in enumerate(inner_cv.split(trainval_idx, labels)):
            sub_train_dataset = Subset(dataset, trainval_idx[sub_train_idx])
            val_dataset = Subset(dataset, trainval_idx[val_idx])

            sub_train_loader = DataLoader(sub_train_dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

            _, _, _, _, val_losses, r2_scores = train_test_MLP(
                input_layer_size=input_layer_size,
                train_dataloader=sub_train_loader,
                test_dataloader=val_loader,
                num_epochs=num_epochs,
                lr=lr,
                dropout=dropout,
                weight_decay=weight_decay,
                trial=trial,
                step_offset=inner_fold * num_epochs,  # unique step range per inner fold
            )
            inner_scores.append(val_losses[-1])
        return np.mean(inner_scores)
    return objective

In [4]:
models = ["scgpt", "hvg", "ion"]
num_cells = len(hvg_dataset)
input_layer_sizes = [512+15, 512+15, 328+15]
input_layer_sizes = dict(zip(models, input_layer_sizes))

best_hyperparameters = {}
num_epochs = 1500
batch_size = 25

shuffled_cell_indices=[]

results_to_store = {"predictions": [], "train_losses": [], "test_losses": [], "r2_scores": []}
results = {model: deepcopy(results_to_store) for model in models}
results["cell type mean"] = {"predictions": [], "r2_scores": []}

ephys_truth=[]


outer_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
optuna.logging.set_verbosity(optuna.logging.FATAL)

for outer_fold, (trainval_idx, test_idx) in enumerate(outer_cv.split(np.arange(num_cells), np.array(cell_embeddings_adata.obs["CellType"]))):
    shuffled_cell_indices.append(test_idx)

    for model in models:
        # HYPERPARAMETER TUNING:
        objective = make_objective(
            trainval_idx=trainval_idx,
            dataset=datasets[model],
            input_layer_size=input_layer_sizes[model],
            num_epochs=num_epochs,
            batch_size=batch_size,
            seed=seed
        )
        study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=seed+outer_fold), pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=250))
        study.optimize(objective, n_trials=100, n_jobs=1, gc_after_trial=True, catch=(ValueError,))
        study_file_name = f"Models/outerfold{outer_fold}_w_types_{model}MLP_tuningstudy.pkl"
        joblib.dump(study, study_file_name)
        best_hyperparameters[model] = study.best_params

        # FINAL TRAINING ON FULL TRAINING SET, USING BEST HYPERPARAMETERS:
        train_subset = Subset(datasets[model], trainval_idx)
        test_subset = Subset(datasets[model], test_idx)

        train_dataloader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
        test_dataloader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

        network, predict_subset, ephys_truth_subset, train_losses, test_losses, r2_scores = train_test_MLP(
            input_layer_size=input_layer_sizes[model],
            train_dataloader=train_dataloader,
            test_dataloader=test_dataloader,
            num_epochs=num_epochs,
            lr=best_hyperparameters[model]["lr"],
            dropout=best_hyperparameters[model]["dropout"],
            weight_decay=best_hyperparameters[model]["weight_decay"]
        )
        network.cpu()
        statedict_file_name = f"Models/outerfold{outer_fold}_w_types_{model}MLP_weights.pth"
        torch.save(network.state_dict(), statedict_file_name)

        # store results
        results[model]["predictions"].append(predict_subset)
        results[model]["train_losses"].append(train_losses)
        results[model]["test_losses"].append(test_losses)
        results[model]["r2_scores"].append(r2_scores)

    ephys_truth.append(ephys_truth_subset)
    print(f"Outer Fold {outer_fold} Best Hyperparameters:")
    for key, value in best_hyperparameters.items():
        print(f"{key}: {value}")
    
    # train/test cell type mean prediction
    trainset_ephys_vals = np.array(ephys_tensor[trainval_idx])
    trainset_cells_types = list(cell_embeddings_adata.obs["CellType"].iloc[trainval_idx])
    trainset_cell_types_to_ephys = {}
    for cells_type, ephys_vals in zip(trainset_cells_types, trainset_ephys_vals):
        if cells_type not in trainset_cell_types_to_ephys:
            trainset_cell_types_to_ephys[cells_type] = []
        trainset_cell_types_to_ephys[cells_type].append(ephys_vals)
    
    for cell_types in trainset_cell_types_to_ephys:
        trainset_cell_types_to_ephys[cell_types] = np.mean(trainset_cell_types_to_ephys[cell_types], axis=0)
    
    celltype_preds_subset = []
    for cells_type in list(cell_embeddings_adata.obs["CellType"].iloc[test_idx]):
        celltype_preds_subset.append(trainset_cell_types_to_ephys[cells_type])
    celltype_preds_subset = np.stack(celltype_preds_subset, axis=0)
    
    results["cell type mean"]["predictions"].append(celltype_preds_subset)
    results["cell type mean"]["r2_scores"].append(r2_score(ephys_truth_subset, celltype_preds_subset, multioutput="raw_values"))

Outer Fold 0 Best Hyperparameters:
scgpt: {'lr': 0.05800995342289887, 'dropout': 9.083149545967631e-06, 'weight_decay': 0.0009752413813729368}
hvg: {'lr': 0.0010139614416844976, 'dropout': 0.017091974816046882, 'weight_decay': 0.006126376640408568}
ion: {'lr': 0.0006350779303915132, 'dropout': 0.02635998346690508, 'weight_decay': 6.50768176523503e-05}


Outer Fold 1 Best Hyperparameters:
scgpt: {'lr': 0.08373872743595681, 'dropout': 0.014953030173442848, 'weight_decay': 3.45297023260184e-05}
hvg: {'lr': 0.002158358735993714, 'dropout': 0.011881928163629218, 'weight_decay': 0.007104226548329916}
ion: {'lr': 0.0010958114113251676, 'dropout': 0.028671015074331375, 'weight_decay': 0.0015151998375406719}


Outer Fold 2 Best Hyperparameters:
scgpt: {'lr': 0.09914328217332496, 'dropout': 0.002526931460137866, 'weight_decay': 0.00010382309145186518}
hvg: {'lr': 0.0007077888268426541, 'dropout': 0.0010335122421547363, 'weight_decay': 0.009194231049463972}
ion: {'lr': 0.0007100340006703885, 'dropout': 0.019598338691132703, 'weight_decay': 2.5922487983784766e-05}


Outer Fold 3 Best Hyperparameters:
scgpt: {'lr': 0.06916701873709903, 'dropout': 0.010875560943083708, 'weight_decay': 0.0002111433792167651}
hvg: {'lr': 0.0014897278510940155, 'dropout': 0.0028244829532686855, 'weight_decay': 0.009696968704019334}
ion: {'lr': 0.0008784140507178085, 'dropout': 0.021607864868313065, 'weight_decay': 0.0018085676190452943}


In [5]:
shuffled_cell_indices = np.concatenate(shuffled_cell_indices)

for model in models:
    results[model]["predictions"] = [tensor.numpy() for tensor in results[model]["predictions"]]
    results[model]["predictions"] = np.vstack(results[model]["predictions"])

    results[model]["train_losses"] = np.array(results[model]["train_losses"])
    results[model]["test_losses"] = np.array(results[model]["test_losses"])
    results[model]["r2_scores"] = np.array(results[model]["r2_scores"])

results["cell type mean"]["predictions"] = np.vstack(results["cell type mean"]["predictions"])

ephys_truth = [tensor.numpy() for tensor in ephys_truth]
ephys_truth = np.vstack(ephys_truth)

In [6]:
np.save('w_types_shuffled_cell_indices.npy', shuffled_cell_indices)
with open('w_types_results.pkl', 'wb') as f:
    pickle.dump(results, f)
np.save('w_types_ephys_truth.npy', ephys_truth)

In [7]:
gc.collect()
torch.cuda.empty_cache()